# Pipeline B (v1): Speech Assessment

**Goal:** extract interpretable speech delivery features from user audio for English speaking practice.

**Focus (v1):** fluency/timing + prosody/energy features using audio only .

**Outputs:** extracted features such as pause count, average pause duration, hesitation ration, pitch statistics, and energy statistics.

## Setup & Imports

In [1]:
import os
from pathlib import Path

import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt

## Configuration

In [2]:
# ===== Config ====-=
TARGET_SR = 16000

# Silence detection settings
FRAME_LENGTH = 2048
HOP_LENGTH = 512
TOP_DB = 30   # lower =stricter silence detection

#Pitch range for human speech (a typical safe oen)
FMIN = 75
FMAX = 400

#minimum pause duration to count as a real pause
MIN_PAUSE_SEC = 0.25

## Audio Loading & Standardization

In [3]:
def load_audio(path, target_sr=TARGET_SR):
    """
    Load audio as mono waveform and resample to target_sr.
    Returns:
        y  -> waveform (numpy array)
        sr -> sample rate
    """
    y, sr = librosa.load(path, sr=target_sr, mono=True)
    return y, sr

## Silence / Speech Detection

In [4]:
def detect_non_silent_intervals(y, sr, top_db=TOP_DB, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH):
    """
    Returns non-silent intervals in samples.
    Each row is [start_sample, end_sample].
    """
    intervals = librosa.effects.split(
        y,
        top_db=top_db,
        frame_length=frame_length,
        hop_length=hop_length
    )
    return intervals

## Timing/ Fluency Feature Extraction

In [5]:
def extract_timing_features(y, sr, intervals):
    '''
    #Extract timing/fluency features from detected speech intervals.
    '''
    total_duration_sec = len(y) / sr

    if len(intervals) == 0:
        return {
            "total_duration_sec": total_duration_sec,
            "speech_duration_sec": 0.0,
            "pause_count": 0,
            "avg_pause_duration_sec": 0.0,
            "hesitation_ratio": 1.0
        }

    # speech duration
    speech_samples = sum(end - start for start, end in intervals)
    speech_duration_sec = speech_samples / sr

    # pauses = gaps between speech intervals
    pause_durations = []
    for i in range(1, len(intervals)):
        prev_end = intervals[i - 1][1]
        curr_start = intervals[i][0]
        pause_sec = (curr_start - prev_end) / sr
        if pause_sec > MIN_PAUSE_SEC:
            pause_durations.append(pause_sec)

    pause_count = len(pause_durations)
    avg_pause_duration_sec = float(np.mean(pause_durations)) if pause_durations else 0.0

    silence_duration_sec = max(total_duration_sec - speech_duration_sec, 0.0)
    hesitation_ratio = silence_duration_sec / total_duration_sec if total_duration_sec > 0 else 0.0

    return {
        "total_duration_sec": float(total_duration_sec),
        "speech_duration_sec": float(speech_duration_sec),
        "pause_count": int(pause_count),
        "avg_pause_duration_sec": float(avg_pause_duration_sec),
        "hesitation_ratio": float(hesitation_ratio)
    }

## Pitch and Energy feature extraction

In [6]:
def extract_pitch_energy_features(y, sr, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH):
    """
    #Extract pitch and energy statistics.
    """

    # =---- Energy (RMS) -----
    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
    energy_mean = float(np.mean(rms))
    energy_std = float(np.std(rms))

    # ----- Pitch (YIN) -----
    f0 = librosa.yin(y, fmin=FMIN, fmax=FMAX, sr=sr, frame_length=frame_length, hop_length=hop_length)

    # Remove invalid or zero values
    f0_valid = f0[np.isfinite(f0)]
    f0_valid = f0_valid[f0_valid > 0]

    if len(f0_valid) == 0:
        pitch_mean_hz = 0.0
        pitch_std_hz = 0.0
        pitch_range_hz = 0.0
    else:
        pitch_mean_hz = float(np.mean(f0_valid))
        pitch_std_hz = float(np.std(f0_valid))
        pitch_range_hz = float(np.max(f0_valid) - np.min(f0_valid))

    return {
        "pitch_mean_hz": pitch_mean_hz,
        "pitch_std_hz": pitch_std_hz,
        "pitch_range_hz": pitch_range_hz,
        "energy_mean": energy_mean,
        "energy_std": energy_std
    }

## Full Pipeline B Feature Extraction function

In [7]:
def analyze_audio_features(path):
    """
    Full Pipeline B v1 feature extraction:
    - load audio
    - detect speech intervals
    - timing features
    - pitch/energy features
    """
    y, sr = load_audio(path)
    intervals = detect_non_silent_intervals(y, sr)

    timing_feats = extract_timing_features(y, sr, intervals)
    pitch_energy_feats = extract_pitch_energy_features(y, sr)

    features = {}
    features.update(timing_feats)
    features.update(pitch_energy_feats)

    return features

## Test on one Audio File

In [146]:
test_audio_path = r"C:\Users\User\Desktop\PB testing audio\Presentation1_Test5_M-L_Monotone.wav"
features = analyze_audio_features(test_audio_path)
features

{'total_duration_sec': 40.56,
 'speech_duration_sec': 40.176,
 'pause_count': 0,
 'avg_pause_duration_sec': 0.0,
 'hesitation_ratio': 0.009467455621301782,
 'pitch_mean_hz': 118.76864023239814,
 'pitch_std_hz': 46.37701371427021,
 'pitch_range_hz': 325.23364485981307,
 'energy_mean': 0.01696978509426117,
 'energy_std': 0.007240659557282925}

### Clearer Output

In [147]:
for k, v in features.items():
    print(f"{k}: {v}")

total_duration_sec: 40.56
speech_duration_sec: 40.176
pause_count: 0
avg_pause_duration_sec: 0.0
hesitation_ratio: 0.009467455621301782
pitch_mean_hz: 118.76864023239814
pitch_std_hz: 46.37701371427021
pitch_range_hz: 325.23364485981307
energy_mean: 0.01696978509426117
energy_std: 0.007240659557282925


## Rule-Based Scoring and Feedback

In [148]:
# ==== F;uency scoring ====
def score_fluency(features):
    """
    Score fluency/timing features using simple rule-based thresholds.
    Returns a score from 1 to 3 and feedback messages.
    """

    score = 0
    feedback = []

    pause_count = features["pause_count"]
    avg_pause = features["avg_pause_duration_sec"]
    hesitation_ratio = features["hesitation_ratio"]

    # --- Pause count ---
    if pause_count <= 4:
        score += 3
        feedback.append("Your use of pauses is generally natural.")
    elif pause_count <= 8:
        score += 2
        feedback.append("Your pauses are noticeable but still within a moderate range.")
    else:
        score += 1
        feedback.append("You pause quite frequently, which may reduce fluency.")

    # --- Average pause duration ---
    if avg_pause <= 0.35:
        score += 3
        feedback.append("Your pauses are mostly short and natural.")
    elif avg_pause <= 0.7:
        score += 2
        feedback.append("Some pauses are slightly long.")
    else:
        score += 1
        feedback.append("Your pauses are often long, which may interrupt flow.")

    # --- Hesitation ratio ---
    if hesitation_ratio <= 0.20:
        score += 3
        feedback.append("Your speech flow is fairly smooth.")
    elif hesitation_ratio <= 0.35:
        score += 2
        feedback.append("There is a moderate amount of silence in your delivery.")
    else:
        score += 1
        feedback.append("A large portion of your speech contains silence or hesitation.")

    avg_score = score / 3.0
    return avg_score, feedback

In [149]:
# ===== Prosody scoring ===
def score_prosody(features):
    """
    Score prosody/energy features using simple rule-based thresholds.
    Returns a score from 1 to 3 and feedback messages.
    """

    score = 0
    feedback = []

    pitch_std = features["pitch_std_hz"]
    pitch_range = features["pitch_range_hz"]
    energy_std = features["energy_std"]

    # --- Pitch variability ---
    if pitch_std >= 40:
        score += 3
        feedback.append("Your pitch variation helps your delivery sound expressive.")
    elif pitch_std >= 20:
        score += 2
        feedback.append("Your pitch variation is moderate.")
    else:
        score += 1
        feedback.append("Your speech may sound monotone due to low pitch variation.")

    # --- Pitch range ---
    if pitch_range >= 150:
        score += 3
        feedback.append("Your pitch range supports an engaging speaking style.")
    elif pitch_range >= 80:
        score += 2
        feedback.append("Your pitch range is acceptable but could be more expressive.")
    else:
        score += 1
        feedback.append("Your pitch range is narrow, which may reduce expressiveness.")

    # --- Energy variability ---
    if 0.01 <= energy_std <= 0.05:
        score += 3
        feedback.append("Your loudness is reasonably well controlled.")
    elif 0.005 <= energy_std < 0.01 or 0.05 < energy_std <= 0.08:
        score += 2
        feedback.append("Your loudness variation is moderate.")
    else:
        score += 1
        feedback.append("Your loudness is either too flat or too inconsistent.")

    avg_score = score / 3.0
    return avg_score, feedback

In [157]:
#==== overall level=====

def score_to_level(score):
    """
    #Convert average score to Low / Medium / High.
    """
    if score >= 2.5:
        return "High"
    elif score >= 1.8:
        return "Medium"
    else:
        return "Low"

In [151]:
# ===== full scoring Function ======

def assess_delivery(features):
    """
    Combine fluency and prosody scoring into one assessment.
    Returns scores, level, and feedback.
    """

    fluency_score, fluency_feedback = score_fluency(features)
    prosody_score, prosody_feedback = score_prosody(features)

    overall_score = (fluency_score + prosody_score) / 2.0
    overall_level = score_to_level(overall_score)

    feedback = fluency_feedback + prosody_feedback

    # Add one short summary line
    if overall_level == "High":
        feedback.insert(0, "Your delivery is generally strong and clear.")
    elif overall_level == "Medium":
        feedback.insert(0, "Your delivery is fairly good but has some areas for improvement.")
    else:
        feedback.insert(0, "Your delivery needs improvement in fluency and/or expressiveness.")

    return {
        "fluency_score": round(fluency_score, 2),
        "prosody_score": round(prosody_score, 2),
        "overall_score": round(overall_score, 2),
        "overall_level": overall_level,
        "feedback": feedback
    }

## Test Rule-Based Assessment

In [152]:
assessment = assess_delivery(features)
assessment

{'fluency_score': 3.0,
 'prosody_score': 2.67,
 'overall_score': 2.83,
 'overall_level': 'High',
 'feedback': ['Your delivery is generally strong and clear.',
  'Your use of pauses is generally natural.',
  'Your pauses are mostly short and natural.',
  'Your speech flow is fairly smooth.',
  'Your pitch variation helps your delivery sound expressive.',
  'Your pitch range supports an engaging speaking style.',
  'Your loudness variation is moderate.']}

In [153]:
#clearer output
print(f"fluency_score: {assessment['fluency_score']}")
print(f"prosody_score: {assessment['prosody_score']}")
print(f"overall_score: {assessment['overall_score']}")
print(f"overall_level: {assessment['overall_level']}")

print("\nfeedback:")
for item in assessment["feedback"]:
    print(f"- {item}")

fluency_score: 3.0
prosody_score: 2.67
overall_score: 2.83
overall_level: High

feedback:
- Your delivery is generally strong and clear.
- Your use of pauses is generally natural.
- Your pauses are mostly short and natural.
- Your speech flow is fairly smooth.
- Your pitch variation helps your delivery sound expressive.
- Your pitch range supports an engaging speaking style.
- Your loudness variation is moderate.


## Combine Feature Extraction and Assessment Output

In [154]:
def run_pipeline_b(path):
    """
    Full Pipeline B v1:
    1. Extract raw delivery features
    2. Assess delivery using rule-based scoring
    3. Return one combined result
    """
    features = analyze_audio_features(path)
    assessment = assess_delivery(features)

    result = {
        "features": features,
        "assessment": assessment
    }

    return result

## Test Full Pipeline B Output

In [155]:
pipeline_b_result = run_pipeline_b(test_audio_path)
pipeline_b_result

{'features': {'total_duration_sec': 40.56,
  'speech_duration_sec': 40.176,
  'pause_count': 0,
  'avg_pause_duration_sec': 0.0,
  'hesitation_ratio': 0.009467455621301782,
  'pitch_mean_hz': 118.76864023239814,
  'pitch_std_hz': 46.37701371427021,
  'pitch_range_hz': 325.23364485981307,
  'energy_mean': 0.01696978509426117,
  'energy_std': 0.007240659557282925},
 'assessment': {'fluency_score': 3.0,
  'prosody_score': 2.67,
  'overall_score': 2.83,
  'overall_level': 'High',
  'feedback': ['Your delivery is generally strong and clear.',
   'Your use of pauses is generally natural.',
   'Your pauses are mostly short and natural.',
   'Your speech flow is fairly smooth.',
   'Your pitch variation helps your delivery sound expressive.',
   'Your pitch range supports an engaging speaking style.',
   'Your loudness variation is moderate.']}}

In [156]:
#Clearer output

print("=== FEATURES ===")
for k, v in pipeline_b_result["features"].items():
    print(f"{k}: {v}")

print("\n=== ASSESSMENT ===")
print(f"fluency_score: {pipeline_b_result['assessment']['fluency_score']}")
print(f"prosody_score: {pipeline_b_result['assessment']['prosody_score']}")
print(f"overall_score: {pipeline_b_result['assessment']['overall_score']}")
print(f"overall_level: {pipeline_b_result['assessment']['overall_level']}")

print("\n=== FEEDBACK ===")
for item in pipeline_b_result["assessment"]["feedback"]:
    print(f"- {item}")

=== FEATURES ===
total_duration_sec: 40.56
speech_duration_sec: 40.176
pause_count: 0
avg_pause_duration_sec: 0.0
hesitation_ratio: 0.009467455621301782
pitch_mean_hz: 118.76864023239814
pitch_std_hz: 46.37701371427021
pitch_range_hz: 325.23364485981307
energy_mean: 0.01696978509426117
energy_std: 0.007240659557282925

=== ASSESSMENT ===
fluency_score: 3.0
prosody_score: 2.67
overall_score: 2.83
overall_level: High

=== FEEDBACK ===
- Your delivery is generally strong and clear.
- Your use of pauses is generally natural.
- Your pauses are mostly short and natural.
- Your speech flow is fairly smooth.
- Your pitch variation helps your delivery sound expressive.
- Your pitch range supports an engaging speaking style.
- Your loudness variation is moderate.
